In [27]:
import pandas as pd
import numpy as np
import os
from glob import glob
from sklearn.metrics import roc_curve, auc, balanced_accuracy_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from scipy.optimize import brentq
from scipy.interpolate import interp1d
import seaborn as sns

In [51]:
def print_metrics(scores, labels):
    
    def compute_cllr(scores, labels):
        scores = np.array(scores)
        labels = np.array(labels)
        tar = scores[labels == 1]
        non = scores[labels == 0]
        #c1 = np.mean(np.log2(1 + np.exp(-tar)))
        #c2 = np.mean(np.log2(1 + np.exp(non)))
        c1 = np.mean(np.logaddexp(0, -tar) / np.log(2))
        c2 = np.mean(np.logaddexp(0, non) / np.log(2))
        return (c1 + c2) / 2
    
    fpr, tpr, thres = roc_curve(labels, scores)
    rocauc = auc(fpr, tpr)
    fnr = 1 - tpr
    eer_func = interp1d(fpr, fnr)
    eer = brentq(lambda x: eer_func(x) - x, 0., 1.)
    cllr = compute_cllr(scores, labels)
    
    preds = (scores >= 0).astype(int)
    
    bacc = balanced_accuracy_score(preds, labels)
    acc = accuracy_score(preds, labels)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return rocauc, eer, cllr, bacc, acc, prec, rec, f1

def analyze_ensemble(all_logLR, labels):
    rocauc, eer, cllr, bacc, acc, prec, rec, f1 = print_metrics(all_logLR, labels)
    
    header = f"{'AUC%':>8} {'EER%':>8} {'Cllr':>8} {'B.Acc.':>8} {'Acc.':>8} {'Prec.':>8} {'Recall':>8} {'F1':>8}"
    print(header)
    print(f"{rocauc*100:>8.2f} "
          f"{eer*100:>8.2f} "
          f"{cllr:>8.4f} "
          f"{bacc*100:>8.2f} "
          f"{acc*100:>8.2f} "
          f"{prec*100:>8.2f} "
          f"{rec*100:>8.2f} "
          f"{f1*100:>8.2f}")

In [58]:
rootpath = "/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv19/experts_trained_on_mix/5_experts_FULL_EVAL"
specific_path = 'A01-A03-A04_1000_pooled_real_from_validation'
#specific_path = 'A01-A03-A04_1000_real_from_validation'
#specific_path = 'ASVspoof2015_1000'
#specific_path = 'Pseudofakes_1000'

full_path = os.path.join(rootpath, specific_path)
print(full_path)

labels = np.load(os.path.join(full_path, "labels.npy"))
logLR_scores = np.load(os.path.join(full_path, "scores_per_expert.npy"))
attack_ids = np.load(os.path.join(full_path, "attack_ids.npy"))
#logit_scores = np.load(os.path.join(full_path, "raw_logits_per_expert.npy"))
#filenames = np.load(os.path.join(full_path, "filenames.npy"))

print()
print("Ensemble performance (PLAIN SUM):")
sum_logLR_scores = np.sum(logLR_scores, axis=0)  # sum expert contributions
analyze_ensemble(sum_logLR_scores, labels)
print()

# Compute individual logLR
unique_attacks = np.unique(attack_ids)
mean_scores_per_attack = {}
for attack in unique_attacks:
    mask = (attack_ids == attack)
    mask = mask.squeeze()  # (1, Nsamples) -> (Nsamples)
    mean_scores_per_attack[attack] = np.mean(logLR_scores[:, mask], axis=1)

# Visualize table
df = pd.DataFrame({a: mean_scores_per_attack[a] for a in unique_attacks})
df.index = [f"Exp {i+1}" for i in range(logLR_scores.shape[0])]
df.round(2)

/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv19/experts_trained_on_mix/5_experts_FULL_EVAL/A01-A03-A04_1000_pooled_real_from_validation

Ensemble performance (PLAIN SUM):
    AUC%     EER%     Cllr   B.Acc.     Acc.    Prec.   Recall       F1
   84.91    23.88   0.7120    60.13    66.33    97.82    63.74    77.19



,-,A01,A02,A03,A04,A07,A08,A09,A10,A11,A12
Exp 1,0.03,0.06,-0.23,-0.14,0.20,-0.26,-0.03,-0.26,-0.03,-0.05,-0.02
Exp 2,-0.08,0.03,0.15,0.24,-0.16,-0.14,0.11,0.20,-0.13,0.07,-0.20
Exp 3,-0.32,0.05,-0.13,0.59,-0.30,0.24,0.54,0.90,0.06,0.19,-0.07
Exp 4,-0.40,-0.64,-0.36,0.52,1.55,-0.25,0.04,1.13,-0.22,1.37,1.86
Exp 5,-0.09,0.36,0.13,0.22,-0.13,0.04,0.35,0.02,-0.09,-0.06,-0.11
